In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change runtime type")

from google.colab import drive, userdata
drive.mount("/content/drive")

tok = userdata.get("GH_TOKEN")
!rm -rf /content/repo
!git clone -q https://{tok}@github.com/mennanasa12-cloud/paradox_perception.git /content/repo
!pip -q install albumentations pyyaml

!mkdir -p /content/data
!cp /content/drive/MyDrive/paradox/lusnar_processed.zip /content/data/
!cd /content/data && unzip -q -o lusnar_processed.zip
!ls /content/data

GPU: Tesla T4
Mounted at /content/drive
lusnar_processed.zip  test  train  val


In [ ]:
import sys; sys.path.insert(0, "/content/repo")
from src.perception.dataset.lusnar_seg import LuSNARSeg
from src.perception.utils.config import load_classes

cfg = load_classes()
print(cfg["segmentation_class_weights"], cfg["segmentation_ignore_index"])
for s in ["train", "val", "test"]:
    print(s, len(LuSNARSeg("/content/data", s)))

{0: 0.244, 1: 1.829, 2: 1.361, 3: 1.158, 4: 0.408} 255
train 1916
val 271
test 418


In [ ]:
!pip -q install transformers
!mkdir -p /content/repo/src/perception/segmentation
!touch /content/repo/src/perception/segmentation/__init__.py

In [ ]:
%%writefile /content/repo/src/perception/segmentation/model.py
import torch
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

def build_model(num_classes=5, pretrained="nvidia/segformer-b0-finetuned-ade-512-512"):
    return SegformerForSemanticSegmentation.from_pretrained(
        pretrained, num_labels=num_classes, ignore_mismatched_sizes=True)

def forward_logits(model, x):
    """x: normalized tensor (B,3,H,W) -> logits (B,C,H,W)."""
    logits = model(pixel_values=x).logits
    return F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)

@torch.no_grad()
def predict(model, x):
    """Class-ID mask (B,H,W). This is the function the ROS 2 node will call."""
    model.eval()
    return forward_logits(model, x).argmax(1)

Writing /content/repo/src/perception/segmentation/model.py


In [ ]:
%%writefile /content/repo/src/perception/segmentation/train.py
import json, time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ..dataset.lusnar_seg import LuSNARSeg
from .model import build_model, forward_logits

def confusion(pred, target, n, ignore=255):
    k = target != ignore
    return torch.bincount(n * target[k] + pred[k], minlength=n * n).reshape(n, n)

def scores(cm):
    inter = cm.diag().float()
    union = cm.sum(0).float() + cm.sum(1).float() - inter
    iou = inter / union.clamp(min=1)
    present = cm.sum(1) > 0            # mIoU over classes that appear in the ground truth
    return iou, iou[present].mean().item()

@torch.no_grad()
def evaluate(model, loader, n, ignore, device="cuda"):
    model.eval()
    cm = torch.zeros(n, n, dtype=torch.long, device=device)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = forward_logits(model, x).argmax(1)
        cm += confusion(pred, y, n, ignore)
    return scores(cm.cpu())

def train(data_root, cfg, out_dir, epochs=30, bs=8, lr=6e-5, workers=2):
    device = "cuda"
    n = len(cfg["segmentation"])
    ignore = cfg["segmentation_ignore_index"]
    names = [cfg["segmentation"][i] for i in range(n)]
    w = torch.tensor([cfg["segmentation_class_weights"][i] for i in range(n)],
                     dtype=torch.float32, device=device)

    tr = DataLoader(LuSNARSeg(data_root, "train"), bs, shuffle=True,
                    num_workers=workers, drop_last=True, pin_memory=True)
    va = DataLoader(LuSNARSeg(data_root, "val"), bs, num_workers=workers)

    model = build_model(n).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(tr))
    loss_fn = nn.CrossEntropyLoss(weight=w, ignore_index=ignore)
    scaler = torch.amp.GradScaler("cuda")

    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    best, log = -1.0, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(forward_logits(model, x).float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        iou, miou = evaluate(model, va, n, ignore, device)
        row = {"epoch": ep, "loss": round(tot / len(tr), 4), "val_mIoU": round(miou, 4),
               "val_IoU": {names[i]: round(iou[i].item(), 3) for i in range(n)}}
        log.append(row); print(f"{time.time()-t0:.0f}s", row)
        if miou > best:
            best = miou
            torch.save(model.state_dict(), out / "best.pt")
        json.dump(log, open(out / "log.json", "w"), indent=2)
    print("best val mIoU:", round(best, 4))
    return model

def test(data_root, cfg, ckpt, bs=8):
    n = len(cfg["segmentation"]); ignore = cfg["segmentation_ignore_index"]
    names = [cfg["segmentation"][i] for i in range(n)]
    model = build_model(n).to("cuda")
    model.load_state_dict(torch.load(ckpt))
    te = DataLoader(LuSNARSeg(data_root, "test"), bs, num_workers=2)
    iou, miou = evaluate(model, te, n, ignore)
    print("TEST mIoU:", round(miou, 4), {names[i]: round(iou[i].item(), 3) for i in range(n)})

Writing /content/repo/src/perception/segmentation/train.py


In [ ]:
import sys; sys.path.insert(0, "/content/repo")
from src.perception.segmentation.train import train
from src.perception.utils.config import load_classes

cfg = load_classes()
_ = train("/content/data", cfg, "/content/drive/MyDrive/paradox/checkpoints/smoke", epochs=2)

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([5])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([5, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


71s {'epoch': 1, 'loss': 0.6783, 'val_mIoU': 0.673, 'val_IoU': {'regolith': 0.943, 'crater': 0.494, 'rock': 0.343, 'mountain': 0.611, 'sky': 0.974}}
56s {'epoch': 2, 'loss': 0.335, 'val_mIoU': 0.6759, 'val_IoU': {'regolith': 0.943, 'crater': 0.467, 'rock': 0.36, 'mountain': 0.635, 'sky': 0.975}}
best val mIoU: 0.6759


In [ ]:
_ = train("/content/data", cfg, "/content/drive/MyDrive/paradox/checkpoints/segformer_b0", epochs=30)

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([5])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([5, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


54s {'epoch': 1, 'loss': 0.6241, 'val_mIoU': 0.6607, 'val_IoU': {'regolith': 0.946, 'crater': 0.518, 'rock': 0.32, 'mountain': 0.545, 'sky': 0.974}}
49s {'epoch': 2, 'loss': 0.2433, 'val_mIoU': 0.7087, 'val_IoU': {'regolith': 0.949, 'crater': 0.482, 'rock': 0.423, 'mountain': 0.712, 'sky': 0.978}}
55s {'epoch': 3, 'loss': 0.161, 'val_mIoU': 0.7381, 'val_IoU': {'regolith': 0.953, 'crater': 0.525, 'rock': 0.449, 'mountain': 0.779, 'sky': 0.984}}
53s {'epoch': 4, 'loss': 0.1321, 'val_mIoU': 0.7681, 'val_IoU': {'regolith': 0.967, 'crater': 0.611, 'rock': 0.509, 'mountain': 0.77, 'sky': 0.985}}
52s {'epoch': 5, 'loss': 0.1098, 'val_mIoU': 0.775, 'val_IoU': {'regolith': 0.966, 'crater': 0.635, 'rock': 0.5, 'mountain': 0.789, 'sky': 0.985}}
51s {'epoch': 6, 'loss': 0.1004, 'val_mIoU': 0.7869, 'val_IoU': {'regolith': 0.968, 'crater': 0.658, 'rock': 0.514, 'mountain': 0.808, 'sky': 0.986}}
52s {'epoch': 7, 'loss': 0.09, 'val_mIoU': 0.7949, 'val_IoU': {'regolith': 0.971, 'crater': 0.676, 'rock':

In [ ]:
from src.perception.segmentation.train import test
test("/content/data", cfg, "/content/drive/MyDrive/paradox/checkpoints/segformer_b0/best.pt")

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([5])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([5, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


TEST mIoU: 0.7509 {'regolith': 0.959, 'crater': 0.403, 'rock': 0.627, 'mountain': 0.779, 'sky': 0.987}


In [ ]:
!mkdir -p /content/drive/MyDrive/paradox/code_backup
!cp /content/repo/src/perception/segmentation/*.py /content/drive/MyDrive/paradox/code_backup/
!ls /content/drive/MyDrive/paradox/code_backup

__init__.py  model.py  train.py


In [ ]:
!cd /content/repo && git config user.name "mennanasa12-cloud"
!cd /content/repo && git config user.email "mennanasa12-cloud@users.noreply.github.com"
!cd /content/repo && git add src/perception/segmentation/
!cd /content/repo && git commit -m "Add SegFormer-B0 terrain segmentation: model, training loop and mIoU eval (val 0.827, test 0.751)"
!cd /content/repo && git push

[master e2be562] Add SegFormer-B0 terrain segmentation: model, training loop and mIoU eval (val 0.827, test 0.751)
 3 files changed, 96 insertions(+)
 create mode 100644 src/perception/segmentation/__init__.py
 create mode 100644 src/perception/segmentation/model.py
 create mode 100644 src/perception/segmentation/train.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 2.38 KiB | 2.38 MiB/s, done.
Total 7 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/mennanasa12-cloud/paradox_perception.git
   806947d..e2be562  master -> master


In [ ]:
import torch
from torch.utils.data import DataLoader
from src.perception.dataset.lusnar_seg import LuSNARSeg
def class_pixels(split, n=5, ignore=255):
    counts = torch.zeros(n, dtype=torch.long)
    for x, y in DataLoader(LuSNARSeg("/content/data", split), 8, num_workers=2):
        y = y[y != ignore]
        counts += torch.bincount(y, minlength=n)
    return counts
for s in ["train", "val", "test"]:
       c = class_pixels(s)
       print(s, (c / c.sum() * 100).round(decimals=2).tolist())

train [68.87000274658203, 1.2300000190734863, 2.2100000381469727, 3.059999942779541, 24.6299991607666]
val [68.98999786376953, 2.049999952316284, 1.4800000190734863, 3.509999990463257, 23.979999542236328]
test [68.02999877929688, 1.340000033378601, 4.429999828338623, 3.559999942779541, 22.639999389648438]


In [2]:

from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -iname "*task2*segmentation*.ipynb"


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/task2_segmentation.ipynb


In [4]:

!ls -la /content/repo
!git -C /content/repo rev-parse --show-toplevel


total 12
drwxr-xr-x 3 root root 4096 Oct 11 08:45 .
drwxr-xr-x 1 root root 4096 Oct 11 08:46 ..
drwxr-xr-x 2 root root 4096 Oct 11 08:46 notebooks
fatal: not a git repository (or any of the parent directories): .git


In [6]:


# Copy the notebook from Google Drive
!cp "/content/drive/MyDrive/Colab Notebooks/task2_segmentation.ipynb" /content/task2_segmentation.ipynb

# Clone the correct GitHub repository
!git clone https://github.com/mennanasa12-cloud/paradox_perception.git /content/paradox_perception


Cloning into '/content/paradox_perception'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 59 (delta 14), reused 55 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 382.24 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [9]:

!git config --global user.name "mennanasa12-cloud"
!git config --global user.email "mennanasa12@gmail.com"
